# Atualizar assets de submissao -- RSNA Knee Abnormality Detection

Ponte entre o notebook de treino (que já rodou e gerou um checkpoint bom,
custa caro re-rodar so por causa de uma mudanca de codigo) e o notebook de
submissao (que roda sem internet, entao nao pode clonar codigo novo do
GitHub na hora).

Este notebook: (1) baixa o codigo MAIS RECENTE do GitHub (internet ON,
rapido, sem GPU), (2) copia o checkpoint ja treinado do Output do notebook
de treino (anexado via kernel_sources -- nao precisa retreinar). O Output
combinado (codigo atualizado + checkpoint) e o que o notebook de submissao
deve anexar.

Configure (menu direito, "Notebook options"):
- **Internet**: On
- **Accelerator**: nenhum necessario
- **Add Data**: anexe o Output do notebook de treino mais recente
  (`rsna-knee-train-v7`)

## 1. Baixar o codigo mais recente

In [ ]:
import os
import socket
import urllib.request
import zipfile

socket.setdefaulttimeout(30)

REPO_ZIP_URL = "https://github.com/andreluizpedroso/rsna-knee-abnormality-detection/archive/refs/heads/main.zip"
REPO_DIR = "/kaggle/working/rsna-knee-abnormality-detection"

zip_path = "/kaggle/working/repo.zip"
print("Baixando repositorio...")
urllib.request.urlretrieve(REPO_ZIP_URL, zip_path)
print("Extraindo...")
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/kaggle/working/")
os.rename("/kaggle/working/rsna-knee-abnormality-detection-main", REPO_DIR)
os.remove(zip_path)
print("Pronto:", REPO_DIR)

## 2. Copiar os checkpoints ja treinados (sem retreinar)

In [ ]:
import shutil
from pathlib import Path

def find_first(name, root="/kaggle/input"):
    for p in Path(root).rglob(name):
        return p
    return None

def find_all(name, root="/kaggle/input"):
    return sorted(Path(root).rglob(name))

print("Conteudo de /kaggle/input:", os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "NAO EXISTE")

# Treino agora roda todos os folds (early stopping + N_FOLDS) -- copia TODOS
# os checkpoints best_fold*.pth encontrados, nao so o fold 0, pra inferencia
# poder ensemblar.
checkpoint_srcs = find_all("best_fold*.pth")
assert len(checkpoint_srcs) > 0, (
    "Nenhum best_fold*.pth encontrado em /kaggle/input -- confira se o "
    "Output do notebook de treino foi anexado."
)
print(f"{len(checkpoint_srcs)} checkpoint(s) encontrado(s):", checkpoint_srcs)

checkpoint_dir = "/kaggle/working/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
for checkpoint_src in checkpoint_srcs:
    shutil.copy(checkpoint_src, checkpoint_dir)
    print("copiado para:", os.path.join(checkpoint_dir, checkpoint_src.name))

## Proximo passo

O Output deste notebook (`/kaggle/working/`) tem o codigo atualizado
(`rsna-knee-abnormality-detection/src/*.py`) e o checkpoint
(`checkpoints/best_fold0.pth`) juntos. E isso que o notebook de submissao
(`03_submit_kaggle.ipynb`) deve anexar via `kernel_sources` -- nao mais o
notebook de treino diretamente.